In [1]:
!pip uninstall -y langchain langchain-community langchain-openai langchain-core langchain-text-splitters langchain-classic langgraph langgraph-prebuilt langgraph-sdk langgraph-checkpoint faiss-cpu numpy openai  pypdf pandas scikit-learn 
!pip cache purge
!pip install -U numpy faiss-cpu pypdf tiktoken openai langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-classic langgraph langgraph-prebuilt langgraph-sdk numpy openai  pypdf pandas scikit-learn 

Found existing installation: langchain 1.4.7
Uninstalling langchain-1.4.7:
  Successfully uninstalled langchain-1.4.7
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-openai 1.7.2
Uninstalling langchain-openai-1.7.2:
  Successfully uninstalled langchain-openai-1.7.2
Found existing installation: langchain-core 1.7.0
Uninstalling langchain-core-1.7.0:
  Successfully uninstalled langchain-core-1.7.0
Found existing installation: langchain-text-splitters 1.1.3
Uninstalling langchain-text-splitters-1.1.3:
  Successfully uninstalled langchain-text-splitters-1.1.3
Found existing installation: langchain-classic 1.0.8
Uninstalling langchain-classic-1.0.8:
  Successfully uninstalled langchain-classic-1.0.8
Found existing installation: langgraph 1.2.16
Uninstalling langgraph-1.2.16:
  Successfully uninstalled langgraph-1.2.16
Found existing installation: langgra

You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


Files removed: 128 (96.7 MB)
Directories removed: 265
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ----------------------- ---------------- 7.3/12.7 MB 36.1 MB/s eta 0:00:01
   ------------------------------------ --- 11.5/12.7 MB 29.6 MB/s eta 0:00:01
   ---------------------------------------  12.6/12.7 MB 21.4 MB/s eta 0:00:01
   ---------------------------------------- 12.7/12.7 MB 20.0 MB/s  0:00:00
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   --- ------------------------------------ 1.3/16.6 MB 7.1 MB/s eta 0:00:03
   ------- -------------------------------- 3.1/16.6 MB 8.1 MB/s eta 0:00:02
   ------------- -------------------------- 5.5/16.6 MB 9.3 MB/s eta 0:00:02
   -------------------- ------------------- 8.7/16.6 MB 10.6 MB/s eta 0:00:01
   --------------------------- ------------ 11.5/16.6 MB 11.4 MB/s eta 0:00:01
   ------------------------------------ --- 14.9/16.6 MB 12.2 MB/s eta 0:00:01
   ------------------------

In [2]:
import pandas as pd

from sklearn.metrics import precision_score, recall_score

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

#from langchain.chains import RetrievalQA

C:\Users\Admin\AppData\Local\Temp\ipykernel_13072\1879401507.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
pdf_path = "HRPolicyManual2023.pdf"   # Replace with your PDF file
loader = PyPDFLoader(pdf_path)
documents = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(documents)

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
vectorstore = FAISS.from_documents(docs, embeddings)

In [6]:
llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [7]:
def self_rag(query):
    # Step 1: Preliminary answer
    preliminary = llm.invoke(f"Answer briefly: {query}")
    
    # Step 2: Reflection
    reflection = llm.invoke(
        f"Question: {query}\nPreliminary Answer: {preliminary}\n"
        "Evaluate confidence (High/Low). If Low, say 'Retrieve'."
    )
    
    # Step 3: Retrieval if needed
    if "Retrieve" in reflection:
        retriever = vectorstore.as_retriever(search_kwargs={"k":3})
        retrieved_docs = retriever.get_relevant_documents(query)
        context = " ".join([doc.page_content for doc in retrieved_docs])
        final_answer = llm.invoke(
            f"Question: {query}\nContext: {context}\n"
            f"Preliminary Answer: {preliminary}\nRefine and ground the answer."
        )
    else:
        final_answer = preliminary
    
    return {"query": query, "preliminary": preliminary,
            "reflection": reflection, "final": final_answer}

In [8]:
result = self_rag("Explain DeepRAG in simple terms.")
print(result)

{'query': 'Explain DeepRAG in simple terms.', 'preliminary': AIMessage(content='DeepRAG is a method that combines two things: retrieving information from a large database (like searching for relevant documents) and generating answers using AI (like ChatGPT). It uses deep learning to find the best information and then uses that information to create better, more accurate answers. In simple terms, DeepRAG helps AI give smarter answers by first looking up helpful facts and then using them to respond.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 18, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-